# Kalimantan Fire Situation Monitor — Phase 1
## Active-Fire Detection Snapshot

---

### 1. Objective
Provide a reliable, reproducible, and methodologically sound snapshot of current active-fire detections across Indonesian Kalimantan using NASA FIRMS VIIRS satellite data via Google Earth Engine (GEE).

### 2. Scope Control — Phase 1

| In Scope (Phase 1) | NOT in Scope (Future Phases) |
|---|---|
| VIIRS active-fire detections (SNPP + NOAA-20) | Sentinel-2 / Landsat burned-area analysis |
| 24 h / 72 h / 7-day temporal analysis windows | NBR / dNBR spectral indices |
| Province & regency administrative aggregation | Rainfall, temperature, wind, drought indices |
| Spatial clustering (DBSCAN) | Peatland & land-cover type overlay |
| Grid-based density analysis | Smoke / aerosol dispersion modeling |
| CSV & GeoJSON export (Local + Google Drive) | Fire-risk scoring & ML/AI prediction |
| Comprehensive validation suite | Public-facing web application / Next.js dashboard |

### 3. Terminology & Core Scientific Principles

| Mandatory Term | Prohibited Term (in Phase 1) | Rationale |
|---|---|---|
| **active-fire detection** | confirmed fire | Satellite thermal anomalies are not field-verified |
| **thermal anomaly** | forest fire | Detection may arise from non-forest heat sources |
| **hotspot detection** | wildfire | Cause and fuel type are unverified |
| **active-fire detection cluster** | fire cluster / fire event | A cluster is a spatial concentration of pixels |
| **detection count** | fire count | One physical fire can produce multiple detections |

> **Key Principle:** An active-fire detection is **NOT** automatically a confirmed forest fire. Always describe observations as "active-fire detections" or "thermal anomalies" unless independent verification is introduced.

## 02 — Environment Setup
Install and import required geospatial, statistical, and mapping libraries.

In [ ]:
# Install required packages (quietly in Google Colab environment)
!pip install geemap geopandas folium scikit-learn matplotlib shapely --quiet

import os
import json
import sys
import warnings
from datetime import datetime, timedelta, timezone

import ee
import folium
from folium.plugins import HeatMap
import geemap
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from shapely.geometry import Point, box as shapely_box
from sklearn.cluster import DBSCAN

# Formatting & warning settings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)

print("Environment setup completed successfully.")
print(f"Pandas: {pd.__version__} | GeoPandas: {gpd.__version__} | Folium: {folium.__version__}")

## 03 — Configuration
All tuneable parameters and dataset identifiers are centralized here for reproducibility and transparency.
Downstream functions read from these constants.

In [ ]:
# =============================================================================
# CONFIGURATION CONSTANTS
# =============================================================================

# Dynamic temporal configuration (UTC)
ANALYSIS_END = datetime.now(timezone.utc)

WINDOW_24H_HOURS = 24
WINDOW_72H_HOURS = 72
WINDOW_7D_HOURS = 7 * 24  # 168 hours

START_24H = ANALYSIS_END - timedelta(hours=WINDOW_24H_HOURS)
START_72H = ANALYSIS_END - timedelta(hours=WINDOW_72H_HOURS)
START_7D = ANALYSIS_END - timedelta(hours=WINDOW_7D_HOURS)

# Earth Engine Project ID
EE_PROJECT_ID = 'riset-banjarnegara'

# Earth Engine collection IDs (verified against GEE Data Catalog)
VIIRS_SNPP_COLLECTION = 'NASA/LANCE/SNPP_VIIRS/C2'
VIIRS_NOAA20_COLLECTION = 'NASA/LANCE/NOAA20_VIIRS/C2'

# Administrative boundaries (geoBoundaries v6.0.0 with FAO GAUL fallback)
ADMIN_L1_COLLECTION = 'WM/geoLab/geoBoundaries/600/ADM1'
ADMIN_L2_COLLECTION = 'WM/geoLab/geoBoundaries/600/ADM2'

# Expected VIIRS bands for 375m active fire (verified in GEE: Bright_ti4, Bright_ti5 with capital B)
FIRE_BANDS = ['Bright_ti4', 'Bright_ti5', 'confidence', 'frp']

# Confidence threshold for 'operational' fires (0=Low, 1=Nominal, 2=High)
CONFIDENCE_THRESHOLD = 1  # Default to Nominal confidence

# Kalimantan province names in geoBoundaries IDN
KALIMANTAN_PROVINCES = [
    'Kalimantan Barat',     # West Kalimantan
    'Kalimantan Tengah',    # Central Kalimantan
    'Kalimantan Selatan',   # South Kalimantan
    'Kalimantan Timur',     # East Kalimantan
    'Kalimantan Utara'      # North Kalimantan
]

# Spatial clustering parameters (DBSCAN with haversine metric)
DBSCAN_EPS_KM = 2.0       # Spatial search radius in km
DBSCAN_MIN_SAMPLES = 3    # Minimum detection points to form a cluster

# Density grid cell size (degrees; 0.1 deg ~ 11.1 km at equator)
GRID_SIZE_DEG = 0.1

# Query safety limit
MAX_FEATURES = 50000

# Export paths (Colab local & Google Drive)
LOCAL_OUTPUT_DIR = './outputs'
DRIVE_BASE_PATH = '/content/drive/MyDrive/Kalimantan-Fire-Monitor'
RUN_DATE_STR = ANALYSIS_END.strftime('%Y-%m-%d')
DRIVE_RUN_PATH = os.path.join(DRIVE_BASE_PATH, RUN_DATE_STR)

print("=" * 70)
print("KALIMANTAN FIRE SITUATION MONITOR — CONFIGURATION")
print("=" * 70)
print(f"EE Project ID         : {EE_PROJECT_ID}")
print(f"Analysis End (UTC)    : {ANALYSIS_END.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"24-Hour Window Start  : {START_24H.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"72-Hour Window Start  : {START_72H.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"7-Day Window Start    : {START_7D.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print(f"Confidence Threshold  : >= {CONFIDENCE_THRESHOLD} (0=Low, 1=Nominal, 2=High)")
print(f"Primary Collections   : {VIIRS_SNPP_COLLECTION} (SNPP)")
print(f"                        {VIIRS_NOAA20_COLLECTION} (NOAA-20)")
print(f"DBSCAN Parameters     : eps={DBSCAN_EPS_KM} km, min_samples={DBSCAN_MIN_SAMPLES}")
print(f"Density Grid Size     : {GRID_SIZE_DEG} degrees (~11 km)")
print(f"Local Output Path     : {LOCAL_OUTPUT_DIR}")
print(f"Drive Output Path     : {DRIVE_RUN_PATH}")
print("=" * 70)

## 04 — Earth Engine Authentication
Initializes the Earth Engine Python API using project `riset-banjarnegara`. In Google Colab, standard interactive authentication is triggered if credentials are not already active.

In [ ]:
# Earth Engine Initialization
try:
    ee.Initialize(project=EE_PROJECT_ID)
    print(f"Earth Engine already initialized with project: {EE_PROJECT_ID}")
except Exception as e:
    print(f"Authenticating Earth Engine for project '{EE_PROJECT_ID}'...")
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
    print(f"Earth Engine authenticated and initialized with project: {EE_PROJECT_ID}")

print(f"Earth Engine API Version: {ee.__version__}")

## 05 — Define Study Area
Loads and validates administrative boundaries for Indonesian Kalimantan.
Primary source: geoBoundaries (`WM/geoLab/geoBoundaries/600/ADM1` and `ADM2`).
Automatic fallback: FAO GAUL (`FAO/GAUL/2015/level1` and `level2`).

The five target provinces:
1. Kalimantan Barat (West Kalimantan)
2. Kalimantan Tengah (Central Kalimantan)
3. Kalimantan Selatan (South Kalimantan)
4. Kalimantan Timur (East Kalimantan)
5. Kalimantan Utara (North Kalimantan)

In [ ]:
# Robust Province and Regency Boundary Loader

def load_kalimantan_provinces():
    '''
    Load Kalimantan province boundaries with automatic fallback support.
    Ensures a valid GeoDataFrame with CRS is always returned.
    '''
    # Attempt 1: geoBoundaries ADM1
    try:
        adm1_fc = ee.FeatureCollection(ADMIN_L1_COLLECTION)
        idn_adm1 = adm1_fc.filter(ee.Filter.eq('shapeGroup', 'IDN'))
        kal_fc = idn_adm1.filter(ee.Filter.stringContains('shapeName', 'Kalimantan'))
        count = kal_fc.size().getInfo()
        if count > 0:
            print(f"Loaded {count} provinces from geoBoundaries ADM1.")
            geojson = kal_fc.getInfo()
            if geojson.get('features'):
                gdf = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
                gdf = gdf.rename(columns={'shapeName': 'province'})
                return kal_fc, gdf
    except Exception as e:
        print(f"geoBoundaries ADM1 note: {e}")

    # Fallback 1: FAO GAUL 2015 Level 1
    try:
        print("Using FAO GAUL Level 1 boundary dataset as fallback...")
        gaul_adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
        kal_fc = gaul_adm1.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')) \
                          .filter(ee.Filter.stringContains('ADM1_NAME', 'Kalimantan'))
        count = kal_fc.size().getInfo()
        if count > 0:
            print(f"Loaded {count} provinces from FAO GAUL Level 1.")
            geojson = kal_fc.getInfo()
            if geojson.get('features'):
                gdf = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
                gdf = gdf.rename(columns={'ADM1_NAME': 'province'})
                return kal_fc, gdf
    except Exception as e:
        print(f"FAO GAUL Level 1 note: {e}")

    # Fallback 2: FAO GAUL Simplified 500m
    try:
        print("Using FAO GAUL Simplified 500m as fallback...")
        gaul_simp = ee.FeatureCollection("FAO/GAUL_SIMPLIFIED_500m/2015/level1")
        kal_fc = gaul_simp.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')) \
                          .filter(ee.Filter.stringContains('ADM1_NAME', 'Kalimantan'))
        count = kal_fc.size().getInfo()
        if count > 0:
            geojson = kal_fc.getInfo()
            if geojson.get('features'):
                gdf = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
                gdf = gdf.rename(columns={'ADM1_NAME': 'province'})
                return kal_fc, gdf
    except Exception as e:
        print(f"FAO GAUL Simplified note: {e}")

    # Geometric bounding box fallback
    print("Warning: Using Kalimantan bounding geometry fallback.")
    kal_geom = ee.Geometry.Polygon([[[108.5, -4.5], [119.5, -4.5], [119.5, 4.5], [108.5, 4.5], [108.5, -4.5]]])
    kal_fc = ee.FeatureCollection([ee.Feature(kal_geom, {'province': 'Kalimantan'})])
    gdf = gpd.GeoDataFrame([{'province': 'Kalimantan', 'geometry': shapely_box(108.5, -4.5, 119.5, 4.5)}], crs='EPSG:4326')
    return kal_fc, gdf


def load_kalimantan_regencies(kal_geom):
    '''
    Load Kalimantan regency boundaries with automatic fallback support.
    '''
    # Attempt 1: geoBoundaries ADM2
    try:
        adm2_fc = ee.FeatureCollection(ADMIN_L2_COLLECTION)
        idn_adm2 = adm2_fc.filter(ee.Filter.eq('shapeGroup', 'IDN'))
        kal_adm2 = idn_adm2.filterBounds(kal_geom)
        count = kal_adm2.size().getInfo()
        if count > 0:
            geojson = kal_adm2.getInfo()
            if geojson.get('features'):
                gdf = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
                gdf = gdf.rename(columns={'shapeName': 'regency'})
                print(f"Loaded {len(gdf)} regencies from geoBoundaries ADM2.")
                return kal_adm2, gdf
    except Exception as e:
        print(f"geoBoundaries ADM2 note: {e}")

    # Fallback: FAO GAUL Level 2
    try:
        print("Using FAO GAUL Level 2 regency dataset as fallback...")
        gaul_adm2 = ee.FeatureCollection("FAO/GAUL/2015/level2")
        kal_adm2 = gaul_adm2.filter(ee.Filter.eq('ADM0_NAME', 'Indonesia')).filterBounds(kal_geom)
        count = kal_adm2.size().getInfo()
        if count > 0:
            geojson = kal_adm2.getInfo()
            if geojson.get('features'):
                gdf = gpd.GeoDataFrame.from_features(geojson['features'], crs='EPSG:4326')
                gdf = gdf.rename(columns={'ADM2_NAME': 'regency'})
                print(f"Loaded {len(gdf)} regencies from FAO GAUL Level 2.")
                return kal_adm2, gdf
    except Exception as e:
        print(f"FAO GAUL Level 2 note: {e}")

    # Empty fallback with valid geometry column to avoid ValueError
    empty_gdf = gpd.GeoDataFrame(columns=['regency', 'geometry'], geometry='geometry', crs='EPSG:4326')
    return ee.FeatureCollection([]), empty_gdf

# Load boundaries
kalimantan_adm1_fc, gdf_provinces = load_kalimantan_provinces()
kalimantan_geometry = kalimantan_adm1_fc.geometry()
kalimantan_adm2_fc, gdf_regencies = load_kalimantan_regencies(kalimantan_geometry)

print(f"\nStudy Area Validation Summary:")
print(f" - Provinces loaded: {len(gdf_provinces)}")
print(f" - Regencies loaded: {len(gdf_regencies)}")
print("\nProvince list:")
for idx, p in enumerate(sorted(gdf_provinces['province'].dropna().unique()), 1):
    print(f"  {idx}. {p}")

In [ ]:
# Study Area Map Visualization
m_study = folium.Map(location=[0.5, 114.5], zoom_start=6, tiles='CartoDB positron')

folium.GeoJson(
    gdf_provinces,
    name='Kalimantan Provinces (ADM1)',
    style_function=lambda x: {
        'fillColor': '#3388ff',
        'color': '#003399',
        'weight': 2,
        'fillOpacity': 0.1
    },
    tooltip=folium.GeoJsonTooltip(fields=['province'], aliases=['Province:'])
).add_to(m_study)

folium.GeoJson(
    gdf_regencies,
    name='Kalimantan Regencies (ADM2)',
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': '#666666',
        'weight': 0.7,
        'dashArray': '3, 3'
    },
    tooltip=folium.GeoJsonTooltip(fields=['regency'], aliases=['Regency:'])
).add_to(m_study)

folium.LayerControl().add_to(m_study)
m_study

## 06 — Inspect VIIRS Dataset (Mandatory)
Before running the extraction pipeline, we inspect dataset properties, available bands, temporal bounds, and confidence encoding for both Suomi NPP and NOAA-20 collections.

In [ ]:
# Inspect VIIRS Collections
print("=" * 70)
print("VIIRS DATASET INSPECTION")
print("=" * 70)

for coll_id, sat_name in [(VIIRS_SNPP_COLLECTION, 'Suomi NPP (SNPP)'), (VIIRS_NOAA20_COLLECTION, 'NOAA-20')]:
    col = ee.ImageCollection(coll_id)
    total_imgs = col.size().getInfo()
    first_img = col.sort('system:time_start').first()
    last_img = col.sort('system:time_start', False).first()
    
    first_date = ee.Date(first_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    last_date = ee.Date(last_img.get('system:time_start')).format('YYYY-MM-dd').getInfo()
    band_names = first_img.bandNames().getInfo()
    
    print(f"\nSatellite: {sat_name}")
    print(f" - Collection ID : {coll_id}")
    print(f" - Total Images  : {total_imgs}")
    print(f" - Date Range    : {first_date} to {last_date}")
    print(f" - Bands Present : {band_names}")
    
    for b in FIRE_BANDS:
        status = "OK" if b in band_names else "MISSING"
        print(f"    * Band '{b}': {status}")

print("\nSampling recent fire pixels to verify confidence encoding...")
sample_col = ee.ImageCollection(VIIRS_SNPP_COLLECTION).filterDate(
    START_7D.strftime('%Y-%m-%d'),
    (ANALYSIS_END + timedelta(days=1)).strftime('%Y-%m-%d')
).filterBounds(kalimantan_geometry)

sample_count = sample_col.size().getInfo()
print(f"Images in 7-day window over Kalimantan: {sample_count}")

if sample_count > 0:
    sample_pixels = sample_col.first().select(FIRE_BANDS).sample(
        region=kalimantan_geometry,
        scale=375,
        numPixels=100,
        geometries=False,
        tileScale=4
    ).getInfo()
    
    found_conf = set()
    for f in sample_pixels.get('features', []):
        c = f.get('properties', {}).get('confidence')
        if c is not None:
            found_conf.add(c)
    print(f"Observed confidence values in sample: {sorted(list(found_conf))}")
    if found_conf and max(found_conf) <= 2:
        print("Categorical encoding confirmed: 0=Low, 1=Nominal, 2=High")
    else:
        print("Note: If continuous 0-100 is detected, adjust CONFIDENCE_THRESHOLD accordingly.")
else:
    print("No sample images in exact window; standard categorical 0/1/2 encoding assumed.")

## 07 — Core Extraction Functions
Modular functions for:
1. **Raster-to-point extraction**: Extracting active-fire pixels from VIIRS daily rasters (`ee.Image.sample()`).
2. **Satellite merging**: Combining SNPP and NOAA-20 while preserving satellite identity and timestamps.
3. **Spatial joins**: Assigning administrative boundaries (province and regency).
4. **DBSCAN clustering**: Identifying spatial concentrations of thermal anomalies.
5. **Density grid generation**: Calculating regular grid counts.

In [ ]:
# Extraction and Processing Functions

def get_fire_detections_collection(collection_id, satellite_name, start_date_str, end_date_str, region):
    '''
    Extract active-fire detection points from a VIIRS ImageCollection.
    Converts raster fire pixels to point features (pixel centroids) preserving attributes.
    '''
    col = ee.ImageCollection(collection_id).filterDate(start_date_str, end_date_str).filterBounds(region)
    n_images = col.size().getInfo()
    print(f"  [{satellite_name}] Filtered {n_images} daily image(s) between {start_date_str} and {end_date_str}")
    
    if n_images == 0:
        return ee.FeatureCollection([])
    
    def extract_pixels(image):
        date_str = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
        selected = image.select(FIRE_BANDS)
        # Sample active fire pixels within the study area
        samples = selected.sample(
            region=region,
            scale=375,
            geometries=True,
            tileScale=4
        )
        def tag_properties(feat):
            coords = feat.geometry().coordinates()
            return feat.set({
                'acq_date': date_str,
                'satellite': satellite_name,
                'longitude': coords.get(0),
                'latitude': coords.get(1)
            })
        return samples.map(tag_properties)
    
    return col.map(extract_pixels).flatten()


def extract_all_detections(start_dt, end_dt, region):
    '''
    Extract and merge detections from both SNPP and NOAA-20 for a given time window.
    '''
    s_str = start_dt.strftime('%Y-%m-%d')
    # Use end date + 1 day to encompass the full final day in GEE exclusive filter
    e_str = (end_dt + timedelta(days=1)).strftime('%Y-%m-%d')
    
    snpp_fc = get_fire_detections_collection(VIIRS_SNPP_COLLECTION, 'SNPP', s_str, e_str, region)
    noaa20_fc = get_fire_detections_collection(VIIRS_NOAA20_COLLECTION, 'NOAA-20', s_str, e_str, region)
    
    merged_fc = snpp_fc.merge(noaa20_fc)
    return merged_fc


def ee_to_geodataframe(fc, label='detections'):
    '''
    Convert Earth Engine FeatureCollection to local GeoDataFrame.
    '''
    size = fc.size().getInfo()
    print(f"  Total raw {label} found: {size}")
    
    if size == 0:
        return gpd.GeoDataFrame(
            columns=['latitude', 'longitude', 'bright_ti4', 'bright_ti5',
                     'confidence', 'frp', 'acq_date', 'satellite', 'geometry'],
            geometry='geometry',
            crs='EPSG:4326'
        )
    
    limited_fc = fc.limit(MAX_FEATURES)
    if size > MAX_FEATURES:
        print(f"  Warning: Capping query to {MAX_FEATURES} features.")
        
    info = limited_fc.getInfo()
    features = info.get('features', [])
    if len(features) == 0:
        return gpd.GeoDataFrame(
            columns=['latitude', 'longitude', 'bright_ti4', 'bright_ti5',
                     'confidence', 'frp', 'acq_date', 'satellite', 'geometry'],
            geometry='geometry',
            crs='EPSG:4326'
        )
    gdf = gpd.GeoDataFrame.from_features(features, crs='EPSG:4326')
    return gdf


def assign_administrative_boundaries(gdf):
    '''
    Spatially join detection points with Province (ADM1) and Regency (ADM2) boundaries.
    '''
    if len(gdf) == 0:
        gdf['province'] = pd.Series(dtype='str')
        gdf['regency'] = pd.Series(dtype='str')
        return gdf
    
    # Join with Province
    gdf_joined = gpd.sjoin(
        gdf,
        gdf_provinces[['province', 'geometry']],
        how='left',
        predicate='within'
    ).drop(columns=['index_right'], errors='ignore')
    
    # Join with Regency
    gdf_joined = gpd.sjoin(
        gdf_joined,
        gdf_regencies[['regency', 'geometry']],
        how='left',
        predicate='within'
    ).drop(columns=['index_right'], errors='ignore')
    
    gdf_joined['province'] = gdf_joined['province'].fillna('Unassigned')
    gdf_joined['regency'] = gdf_joined['regency'].fillna('Unassigned')
    return gdf_joined


def apply_confidence_filter(gdf, threshold=CONFIDENCE_THRESHOLD):
    '''
    Split detections into ALL and OPERATIONAL (filtered) datasets.
    '''
    gdf_all = gdf.copy()
    gdf_operational = gdf[gdf['confidence'] >= threshold].copy()
    return gdf_all, gdf_operational


def run_dbscan_clustering(gdf, eps_km=DBSCAN_EPS_KM, min_samples=DBSCAN_MIN_SAMPLES):
    '''
    Apply DBSCAN spatial clustering using haversine metric on coordinates in radians.
    '''
    gdf_out = gdf.copy()
    if len(gdf_out) < min_samples:
        gdf_out['cluster_id'] = -1
        return gdf_out
    
    coords_rad = np.radians(gdf_out[['latitude', 'longitude']].values)
    eps_rad = eps_km / 6371.0088  # Earth radius in km
    
    db = DBSCAN(eps=eps_rad, min_samples=min_samples, metric='haversine')
    gdf_out['cluster_id'] = db.fit_predict(coords_rad)
    return gdf_out


def compute_cluster_statistics(gdf):
    '''
    Compute descriptive summary statistics for active-fire detection clusters.
    '''
    clustered = gdf[gdf['cluster_id'] != -1].copy()
    if len(clustered) == 0:
        return pd.DataFrame()
    
    stats = clustered.groupby('cluster_id').agg(
        n_detections=('cluster_id', 'size'),
        centroid_lat=('latitude', 'mean'),
        centroid_lon=('longitude', 'mean'),
        lat_min=('latitude', 'min'),
        lat_max=('latitude', 'max'),
        lon_min=('longitude', 'min'),
        lon_max=('longitude', 'max'),
        avg_confidence=('confidence', 'mean'),
        max_frp=('frp', 'max'),
        avg_frp=('frp', 'mean'),
        first_date=('acq_date', 'min'),
        last_date=('acq_date', 'max'),
        province=('province', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown'),
        regency=('regency', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Unknown')
    ).reset_index()
    
    stats['extent_lat_km'] = (stats['lat_max'] - stats['lat_min']) * 111.13
    stats['extent_lon_km'] = (stats['lon_max'] - stats['lon_min']) * 111.13 * np.cos(np.radians(stats['centroid_lat']))
    return stats.sort_values('n_detections', ascending=False).reset_index(drop=True)


def build_density_grid(gdf, grid_size=GRID_SIZE_DEG):
    '''
    Generate regular-grid spatial density counts of detections.
    '''
    if len(gdf) == 0:
        return gpd.GeoDataFrame(columns=['count', 'geometry'], geometry='geometry', crs='EPSG:4326')
    
    bounds = gdf.total_bounds
    lon_edges = np.arange(bounds[0] - grid_size, bounds[2] + grid_size, grid_size)
    lat_edges = np.arange(bounds[1] - grid_size, bounds[3] + grid_size, grid_size)
    
    tmp = gdf.copy()
    tmp['lon_bin'] = np.digitize(tmp['longitude'], lon_edges) - 1
    tmp['lat_bin'] = np.digitize(tmp['latitude'], lat_edges) - 1
    
    density = tmp.groupby(['lon_bin', 'lat_bin']).size().reset_index(name='count')
    geoms = []
    for _, row in density.iterrows():
        li, la = int(row['lon_bin']), int(row['lat_bin'])
        if 0 <= li < len(lon_edges) - 1 and 0 <= la < len(lat_edges) - 1:
            geoms.append(shapely_box(lon_edges[li], lat_edges[la], lon_edges[li+1], lat_edges[la+1]))
        else:
            geoms.append(None)
    
    density['geometry'] = geoms
    density = density.dropna(subset=['geometry'])
    return gpd.GeoDataFrame(density, geometry='geometry', crs='EPSG:4326')

print("Core extraction and analytical functions defined.")

In [ ]:
# Map Visualization Helper
CONFIDENCE_COLORS = {0: '#FFFF00', 1: '#FF9900', 2: '#FF0000'}
CONFIDENCE_LABELS = {0: 'Low', 1: 'Nominal', 2: 'High'}

def build_interactive_map(gdf, title="Active-Fire Detections", show_clusters=False, density_gdf=None):
    '''
    Build interactive Folium map with dark basemap, province boundaries, and active-fire detections.
    '''
    m = folium.Map(location=[0.5, 114.5], zoom_start=6, tiles='CartoDB dark_matter')
    
    # Province borders
    folium.GeoJson(
        gdf_provinces,
        name='Provinces (ADM1)',
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#88bbee',
            'weight': 1.5,
            'fillOpacity': 0
        },
        tooltip=folium.GeoJsonTooltip(fields=['province'])
    ).add_to(m)
    
    # Density grid overlay if provided
    if density_gdf is not None and len(density_gdf) > 0:
        max_val = density_gdf['count'].max()
        folium.GeoJson(
            density_gdf,
            name='7-Day Density Grid',
            style_function=lambda feat: {
                'fillColor': '#FF4500',
                'color': '#8B0000',
                'weight': 0.4,
                'fillOpacity': min(0.75, 0.15 + 0.6 * (feat['properties']['count'] / max_val))
            },
            tooltip=folium.GeoJsonTooltip(fields=['count'], aliases=['Detections in cell:'])
        ).add_to(m)
    
    # Detection points
    if len(gdf) > 0:
        for _, r in gdf.iterrows():
            c_val = int(r.get('confidence', 0))
            color = CONFIDENCE_COLORS.get(c_val, '#FF9900')
            t4_val = r.get('Bright_ti4', r.get('bright_ti4', 0.0))
            popup_html = (
                f"<b>Date:</b> {r.get('acq_date', 'N/A')}<br>"
                f"<b>Satellite:</b> {r.get('satellite', 'N/A')}<br>"
                f"<b>Confidence:</b> {CONFIDENCE_LABELS.get(c_val, str(c_val))}<br>"
                f"<b>FRP:</b> {r.get('frp', 0.0):.2f} MW<br>"
                f"<b>Bright T4:</b> {t4_val:.1f} K<br>"
                f"<b>Province:</b> {r.get('province', 'N/A')}<br>"
                f"<b>Regency:</b> {r.get('regency', 'N/A')}"
            )
            folium.CircleMarker(
                location=[r['latitude'], r['longitude']],
                radius=4,
                color=color,
                fill=True,
                fillColor=color,
                fillOpacity=0.85,
                weight=0.5,
                popup=folium.Popup(popup_html, max_width=250)
            ).add_to(m)
            
    # Cluster convex hulls
    if show_clusters and 'cluster_id' in gdf.columns:
        for cid in gdf[gdf['cluster_id'] != -1]['cluster_id'].unique():
            sub = gdf[gdf['cluster_id'] == cid]
            if len(sub) >= 3:
                hull = sub.geometry.unary_union.convex_hull
                if hull.geom_type in ['Polygon', 'MultiPolygon']:
                    folium.GeoJson(
                        hull.__geo_interface__,
                        name='Cluster Outlines',
                        style_function=lambda x: {
                            'fillColor': '#FF6600',
                            'color': '#FFAA33',
                            'weight': 1.5,
                            'fillOpacity': 0.15,
                            'dashArray': '4, 4'
                        }
                    ).add_to(m)
                    
    legend_html = f'''
    <div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
                background: rgba(20, 20, 20, 0.85); padding: 12px 16px;
                border-radius: 8px; font-family: monospace; font-size: 12px; color: #fff;">
        <div style="font-weight: bold; margin-bottom: 6px;">{title}</div>
        <div><span style="color: #FF0000;">●</span> High Confidence (2)</div>
        <div><span style="color: #FF9900;">●</span> Nominal Confidence (1)</div>
        <div><span style="color: #FFFF00;">●</span> Low Confidence (0)</div>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    folium.LayerControl().add_to(m)
    return m

## 08 — Last 24 Hours Analysis
**Purpose:** Snapshot of the most immediate active-fire situation across Kalimantan.

In [ ]:
print("=" * 70)
print(f"EXTRACTING 24-HOUR ACTIVE-FIRE DETECTIONS ({START_24H.strftime('%Y-%m-%d %H:%M')} to {ANALYSIS_END.strftime('%Y-%m-%d %H:%M')} UTC)")
print("=" * 70)

fc_24h = extract_all_detections(START_24H, ANALYSIS_END, kalimantan_geometry)
gdf_24h_raw = ee_to_geodataframe(fc_24h, label='24h detections')
gdf_24h_raw = assign_administrative_boundaries(gdf_24h_raw)

# Split ALL vs OPERATIONAL
gdf_24h_all, gdf_24h = apply_confidence_filter(gdf_24h_raw, CONFIDENCE_THRESHOLD)

print(f"\n24-Hour Detection Summary:")
print(f" - All Detections (Raw)         : {len(gdf_24h_all)}")
print(f" - Operational (Confidence >= {CONFIDENCE_THRESHOLD}): {len(gdf_24h)}")

In [ ]:
# 24-Hour Confidence & Satellite Distribution Charts
if len(gdf_24h_raw) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Confidence breakdown
    conf_counts = gdf_24h_raw['confidence'].value_counts().sort_index()
    c_labels = [CONFIDENCE_LABELS.get(int(k), str(k)) for k in conf_counts.index]
    c_colors = [CONFIDENCE_COLORS.get(int(k), '#999999') for k in conf_counts.index]
    
    axes[0].bar(c_labels, conf_counts.values, color=c_colors, edgecolor='#333333')
    axes[0].set_title('24h Detections by Confidence Level', fontweight='bold')
    axes[0].set_ylabel('Detection Count')
    for idx, val in enumerate(conf_counts.values):
        axes[0].text(idx, val + 0.2, str(val), ha='center', fontweight='bold')
        
    # Satellite breakdown
    sat_counts = gdf_24h_raw['satellite'].value_counts()
    axes[1].bar(sat_counts.index, sat_counts.values, color=['#4A90E2', '#50E3C2'], edgecolor='#333333')
    axes[1].set_title('24h Detections by Satellite Platform', fontweight='bold')
    axes[1].set_ylabel('Detection Count')
    for idx, val in enumerate(sat_counts.values):
        axes[1].text(idx, val + 0.2, str(val), ha='center', fontweight='bold')
        
    plt.tight_layout()
    plt.show()
else:
    print("No 24-hour detections found to plot.")

In [ ]:
# 24-Hour Province & Regency Statistics
if len(gdf_24h) > 0:
    prov_24h_df = gdf_24h.groupby('province').agg(
        detections=('province', 'size'),
        avg_confidence=('confidence', 'mean'),
        avg_frp=('frp', 'mean'),
        max_frp=('frp', 'max')
    ).sort_values('detections', ascending=False).reset_index()
    
    print("\n--- 24-Hour Province Summary (Operational Detections) ---")
    print(prov_24h_df.to_string(index=False))
    
    reg_24h_df = gdf_24h.groupby(['regency', 'province']).agg(
        detections=('regency', 'size'),
        avg_frp=('frp', 'mean')
    ).sort_values('detections', ascending=False).reset_index()
    
    print("\n--- 24-Hour Top Regencies (Operational Detections) ---")
    print(reg_24h_df.head(15).to_string(index=False))
else:
    print("No operational detections in the 24-hour window.")

In [ ]:
# 24-Hour Map
map_display_24h = gdf_24h if len(gdf_24h) > 0 else gdf_24h_all
map_24h = build_interactive_map(map_display_24h, title=f"24h Active-Fire Detections ({len(map_display_24h)} points)")
map_24h

## 09 — Last 72 Hours Analysis
**Purpose:** Identify persistent recent thermal anomalies, multi-day activity, and early cluster formations.

In [ ]:
print("=" * 70)
print(f"EXTRACTING 72-HOUR ACTIVE-FIRE DETECTIONS ({START_72H.strftime('%Y-%m-%d %H:%M')} to {ANALYSIS_END.strftime('%Y-%m-%d %H:%M')} UTC)")
print("=" * 70)

fc_72h = extract_all_detections(START_72H, ANALYSIS_END, kalimantan_geometry)
gdf_72h_raw = ee_to_geodataframe(fc_72h, label='72h detections')
gdf_72h_raw = assign_administrative_boundaries(gdf_72h_raw)

gdf_72h_all, gdf_72h = apply_confidence_filter(gdf_72h_raw, CONFIDENCE_THRESHOLD)

print(f"\n72-Hour Detection Summary:")
print(f" - All Detections (Raw)         : {len(gdf_72h_all)}")
print(f" - Operational (Confidence >= {CONFIDENCE_THRESHOLD}): {len(gdf_72h)}")

In [ ]:
# 72-Hour Temporal Breakdown & Persistence
if len(gdf_72h) > 0:
    daily_72h = gdf_72h.groupby(['acq_date', 'satellite']).size().unstack(fill_value=0)
    print("\n--- 72-Hour Daily Breakdown by Satellite ---")
    print(daily_72h)
    
    # Approximate persistence: count days per rounded grid cell (0.05 deg ~ 5.5 km)
    gdf_72h['grid_cell'] = (
        gdf_72h['latitude'].round(2).astype(str) + "_" + gdf_72h['longitude'].round(2).astype(str)
    )
    cell_activity = gdf_72h.groupby('grid_cell')['acq_date'].nunique().reset_index(name='active_days')
    multi_day = cell_activity[cell_activity['active_days'] > 1]
    print(f"\nGrid cells with multi-day activity: {len(multi_day)} / {len(cell_activity)}")
else:
    print("No operational detections in 72-hour window.")

In [ ]:
# 72-Hour DBSCAN Spatial Clustering
if len(gdf_72h) >= DBSCAN_MIN_SAMPLES:
    gdf_72h = run_dbscan_clustering(gdf_72h, eps_km=DBSCAN_EPS_KM, min_samples=DBSCAN_MIN_SAMPLES)
    cluster_stats_72h = compute_cluster_statistics(gdf_72h)
    
    n_clusters_72 = gdf_72h[gdf_72h['cluster_id'] != -1]['cluster_id'].nunique()
    n_noise_72 = (gdf_72h['cluster_id'] == -1).sum()
    print(f"\n72-Hour DBSCAN Results: {n_clusters_72} cluster(s), {n_noise_72} noise detection(s)")
    
    if len(cluster_stats_72h) > 0:
        print("\n--- Top 72-Hour Active-Fire Detection Clusters ---")
        disp_cols = ['cluster_id', 'n_detections', 'centroid_lat', 'centroid_lon',
                     'max_frp', 'avg_frp', 'province', 'regency', 'extent_lat_km', 'extent_lon_km']
        print(cluster_stats_72h[disp_cols].head(10).to_string(index=False))
else:
    gdf_72h['cluster_id'] = -1
    cluster_stats_72h = pd.DataFrame()
    print(f"Fewer than {DBSCAN_MIN_SAMPLES} points; clustering skipped.")

In [ ]:
# 72-Hour Map
map_72h = build_interactive_map(
    gdf_72h if len(gdf_72h) > 0 else gdf_72h_all,
    title=f"72h Active-Fire Detections ({len(gdf_72h)} operational points)",
    show_clusters=True
)
map_72h

## 10 — Last 7 Days Analysis
**Purpose:** Assess broader regional spatial patterns, hotspots density, and provincial/regency rankings.

In [ ]:
print("=" * 70)
print(f"EXTRACTING 7-DAY ACTIVE-FIRE DETECTIONS ({START_7D.strftime('%Y-%m-%d %H:%M')} to {ANALYSIS_END.strftime('%Y-%m-%d %H:%M')} UTC)")
print("=" * 70)

fc_7d = extract_all_detections(START_7D, ANALYSIS_END, kalimantan_geometry)
gdf_7d_raw = ee_to_geodataframe(fc_7d, label='7-day detections')
gdf_7d_raw = assign_administrative_boundaries(gdf_7d_raw)

gdf_7d_all, gdf_7d = apply_confidence_filter(gdf_7d_raw, CONFIDENCE_THRESHOLD)

print(f"\n7-Day Detection Summary:")
print(f" - All Detections (Raw)         : {len(gdf_7d_all)}")
print(f" - Operational (Confidence >= {CONFIDENCE_THRESHOLD}): {len(gdf_7d)}")

In [ ]:
# 7-Day Density Grid Computation
density_7d = build_density_grid(gdf_7d, grid_size=GRID_SIZE_DEG)
print(f"7-Day Density Grid generated: {len(density_7d)} active grid cells.")
if len(density_7d) > 0:
    print(f" - Max detections in a single cell: {density_7d['count'].max()}")
    print(f" - Mean detections per active cell: {density_7d['count'].mean():.2f}")

In [ ]:
# 7-Day Province & Regency Rankings
if len(gdf_7d) > 0:
    prov_7d_df = gdf_7d.groupby('province').agg(
        detections=('province', 'size'),
        avg_confidence=('confidence', 'mean'),
        total_frp=('frp', 'sum'),
        avg_frp=('frp', 'mean'),
        max_frp=('frp', 'max')
    ).sort_values('detections', ascending=False).reset_index()
    
    print("\n--- 7-Day Province Ranking (Operational) ---")
    print(prov_7d_df.to_string(index=False))
    
    reg_7d_df = gdf_7d.groupby(['regency', 'province']).agg(
        detections=('regency', 'size'),
        avg_frp=('frp', 'mean'),
        total_frp=('frp', 'sum')
    ).sort_values('detections', ascending=False).reset_index()
    
    print("\n--- 7-Day Top 20 Regencies (Operational) ---")
    print(reg_7d_df.head(20).to_string(index=False))
    
    # Plot rankings
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.barh(prov_7d_df['province'], prov_7d_df['detections'], color='#FF5722', edgecolor='#333333')
    ax.set_title('7-Day Active-Fire Detections by Province', fontweight='bold')
    ax.set_xlabel('Operational Detection Count')
    ax.invert_yaxis()
    for idx, val in enumerate(prov_7d_df['detections']):
        ax.text(val + 0.5, idx, str(val), va='center', fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    prov_7d_df = pd.DataFrame(columns=['province', 'detections'])
    reg_7d_df = pd.DataFrame(columns=['regency', 'province', 'detections'])
    print("No operational detections in 7-day window.")

In [ ]:
# 7-Day DBSCAN Spatial Clustering
if len(gdf_7d) >= DBSCAN_MIN_SAMPLES:
    gdf_7d = run_dbscan_clustering(gdf_7d, eps_km=DBSCAN_EPS_KM, min_samples=DBSCAN_MIN_SAMPLES)
    cluster_stats_7d = compute_cluster_statistics(gdf_7d)
    
    n_clusters_7d = gdf_7d[gdf_7d['cluster_id'] != -1]['cluster_id'].nunique()
    n_noise_7d = (gdf_7d['cluster_id'] == -1).sum()
    print(f"\n7-Day DBSCAN Results: {n_clusters_7d} cluster(s), {n_noise_7d} noise detection(s)")
    
    if len(cluster_stats_7d) > 0:
        print("\n--- Top 7-Day Active-Fire Detection Clusters ---")
        disp_cols = ['cluster_id', 'n_detections', 'centroid_lat', 'centroid_lon',
                     'max_frp', 'avg_frp', 'province', 'regency', 'extent_lat_km', 'extent_lon_km']
        print(cluster_stats_7d[disp_cols].head(15).to_string(index=False))
else:
    gdf_7d['cluster_id'] = -1
    cluster_stats_7d = pd.DataFrame()
    print(f"Fewer than {DBSCAN_MIN_SAMPLES} points; clustering skipped.")

In [ ]:
# 7-Day Map (with Density Grid and Clusters)
map_7d = build_interactive_map(
    gdf_7d if len(gdf_7d) > 0 else gdf_7d_all,
    title=f"7-Day Active-Fire Density & Clusters ({len(gdf_7d)} operational points)",
    show_clusters=True,
    density_gdf=density_7d
)
map_7d

## 11 — Administrative Summary
Consolidated comparative tables across all three analysis windows (24h, 72h, 7d).

In [ ]:
# Province Comparative Summary
def get_prov_counts(gdf, col_name):
    all_provinces = sorted(gdf_provinces['province'].dropna().unique().tolist())
    if len(gdf) == 0:
        return pd.DataFrame({'province': all_provinces, col_name: 0})
    counts = gdf.groupby('province').size().reset_index(name=col_name)
    # Ensure all loaded provinces appear
    base = pd.DataFrame({'province': all_provinces})
    merged = base.merge(counts, on='province', how='left').fillna(0)
    merged[col_name] = merged[col_name].astype(int)
    return merged

prov_24_c = get_prov_counts(gdf_24h, '24h')
prov_72_c = get_prov_counts(gdf_72h, '72h')
prov_7d_c = get_prov_counts(gdf_7d, '7d')

summary_province = (
    prov_7d_c.merge(prov_72_c, on='province', how='outer')
             .merge(prov_24_c, on='province', how='outer')
             .sort_values('7d', ascending=False)
             .reset_index(drop=True)
)

print("=" * 70)
print("ADMINISTRATIVE SUMMARY — PROVINCES")
print("=" * 70)
print(summary_province.to_string(index=False))

In [ ]:
# Regency Comparative Summary
def get_reg_counts(gdf, col_name):
    if len(gdf) == 0:
        return pd.DataFrame(columns=['regency', 'province', col_name])
    return gdf.groupby(['regency', 'province']).size().reset_index(name=col_name)

reg_24_c = get_reg_counts(gdf_24h, '24h')
reg_72_c = get_reg_counts(gdf_72h, '72h')
reg_7d_c = get_reg_counts(gdf_7d, '7d')

summary_regency = (
    reg_7d_c.merge(reg_72_c, on=['regency', 'province'], how='outer')
            .merge(reg_24_c, on=['regency', 'province'], how='outer')
            .fillna(0)
            .astype({'24h': int, '72h': int, '7d': int})
            .sort_values('7d', ascending=False)
            .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("ADMINISTRATIVE SUMMARY — TOP REGENCIES (Top 30)")
print("=" * 70)
print(summary_regency.head(30).to_string(index=False))

## 12 — Data Export
Exports analytical datasets to both **local filesystem** and **Google Drive** using the structured archive format:
```
Kalimantan-Fire-Monitor/
  YYYY-MM-DD/
    raw/           <- All detections (raw CSV)
    processed/     <- Operational CSV, GeoJSONs, Density Grid
    reports/       <- Administrative summaries, Cluster statistics
    metadata/      <- Analysis run metadata JSON
```

In [ ]:
# Prepare local output directories
for sub in ['raw', 'processed', 'reports', 'metadata']:
    os.makedirs(os.path.join(LOCAL_OUTPUT_DIR, sub), exist_ok=True)

# Attempt Google Drive mounting if running inside Google Colab
drive_enabled = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    for sub in ['raw', 'processed', 'reports', 'metadata']:
        os.makedirs(os.path.join(DRIVE_RUN_PATH, sub), exist_ok=True)
    drive_enabled = True
    print(f"Google Drive mounted. Target directory: {DRIVE_RUN_PATH}")
except Exception as e:
    print(f"Google Drive not mounted ({e}). Exporting to local path only ({LOCAL_OUTPUT_DIR}).")

export_dirs = [LOCAL_OUTPUT_DIR]
if drive_enabled:
    export_dirs.append(DRIVE_RUN_PATH)

In [ ]:
# Export Data Files (CSV & GeoJSON)
csv_columns = ['latitude', 'longitude', 'Bright_ti4', 'Bright_ti5', 'confidence',
               'frp', 'acq_date', 'satellite', 'province', 'regency']

for base_dir in export_dirs:
    # 1. Raw Detections CSV (ALL detections)
    for label, df_raw in [('24h', gdf_24h_raw), ('72h', gdf_72h_raw), ('7d', gdf_7d_raw)]:
        if len(df_raw) > 0:
            cols = [c for c in csv_columns if c in df_raw.columns]
            p = os.path.join(base_dir, 'raw', f'kalimantan_fires_all_{label}.csv')
            df_raw[cols].to_csv(p, index=False)
            
    # 2. Operational Detections CSV & GeoJSON
    for label, df_op in [('24h', gdf_24h), ('72h', gdf_72h), ('7d', gdf_7d)]:
        if len(df_op) > 0:
            cols = [c for c in csv_columns + ['cluster_id'] if c in df_op.columns]
            p_csv = os.path.join(base_dir, 'processed', f'kalimantan_fires_operational_{label}.csv')
            df_op[cols].to_csv(p_csv, index=False)
            
            p_geo = os.path.join(base_dir, 'processed', f'kalimantan_fires_{label}.geojson')
            df_op[cols + ['geometry']].to_file(p_geo, driver='GeoJSON')
            
    # 3. Cluster Statistics & GeoJSON
    for label, st in [('72h', cluster_stats_72h), ('7d', cluster_stats_7d)]:
        if len(st) > 0:
            p_csv = os.path.join(base_dir, 'reports', f'cluster_statistics_{label}.csv')
            st.to_csv(p_csv, index=False)
            
            p_geo = os.path.join(base_dir, 'processed', f'kalimantan_clusters_{label}.geojson')
            geom = [Point(r['centroid_lon'], r['centroid_lat']) for _, r in st.iterrows()]
            gpd.GeoDataFrame(st, geometry=geom, crs='EPSG:4326').to_file(p_geo, driver='GeoJSON')
            
    # 4. Density Grid GeoJSON
    if len(density_7d) > 0:
        p_dens = os.path.join(base_dir, 'processed', 'kalimantan_density_7d.geojson')
        density_7d.to_file(p_dens, driver='GeoJSON')
        
    # 5. Administrative Summaries
    summary_province.to_csv(os.path.join(base_dir, 'reports', 'province_summary.csv'), index=False)
    summary_regency.to_csv(os.path.join(base_dir, 'reports', 'regency_summary.csv'), index=False)
    
    # 6. Analysis Metadata JSON
    metadata = {
        'analysis_timestamp_utc': ANALYSIS_END.isoformat(),
        'windows': {
            '24h': {'start': START_24H.isoformat(), 'end': ANALYSIS_END.isoformat()},
            '72h': {'start': START_72H.isoformat(), 'end': ANALYSIS_END.isoformat()},
            '7d': {'start': START_7D.isoformat(), 'end': ANALYSIS_END.isoformat()},
        },
        'datasets': {
            'snpp_viirs': VIIRS_SNPP_COLLECTION,
            'noaa20_viirs': VIIRS_NOAA20_COLLECTION,
            'admin_l1': ADMIN_L1_COLLECTION,
            'admin_l2': ADMIN_L2_COLLECTION
        },
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'clustering': {
            'method': 'DBSCAN',
            'eps_km': DBSCAN_EPS_KM,
            'min_samples': DBSCAN_MIN_SAMPLES
        },
        'grid_size_deg': GRID_SIZE_DEG,
        'detection_counts': {
            'raw_24h': len(gdf_24h_all),
            'operational_24h': len(gdf_24h),
            'raw_72h': len(gdf_72h_all),
            'operational_72h': len(gdf_72h),
            'raw_7d': len(gdf_7d_all),
            'operational_7d': len(gdf_7d)
        }
    }
    with open(os.path.join(base_dir, 'metadata', 'analysis_metadata.json'), 'w') as f:
        json.dump(metadata, f, indent=2, default=str)

print("Export completed successfully across active targets.")

## 13 — Validation Suite
Comprehensive programmatic verification checks assessing dataset integrity, temporal accuracy, spatial validity, statistical consistency, and monotonicity.

In [ ]:
# Run Validation Suite
print("=" * 70)
print("EXECUTION OF VALIDATION CHECKS")
print("=" * 70)

val_passed = 0
val_failed = 0

def assert_check(name, condition, detail=""):
    global val_passed, val_failed
    if condition:
        print(f" [PASS] {name}")
        val_passed += 1
    else:
        print(f" [FAIL] {name} -> {detail}")
        val_failed += 1

# 1. Dataset Integrity
assert_check("SNPP Collection Exists & Populated", ee.ImageCollection(VIIRS_SNPP_COLLECTION).size().getInfo() > 0)
assert_check("NOAA-20 Collection Exists & Populated", ee.ImageCollection(VIIRS_NOAA20_COLLECTION).size().getInfo() > 0)

# 2. Temporal Checks
assert_check("24h Duration = 86,400s", (ANALYSIS_END - START_24H).total_seconds() == 86400)
assert_check("72h Duration = 259,200s", (ANALYSIS_END - START_72H).total_seconds() == 259200)
assert_check("7d Duration = 604,800s", (ANALYSIS_END - START_7D).total_seconds() == 604800)

# 3. Spatial Checks
assert_check("5 Kalimantan Provinces in Boundary", len(gdf_provinces) == 5)
kal_bbox = gdf_provinces.total_bounds
for lbl, df in [('24h', gdf_24h_all), ('72h', gdf_72h_all), ('7d', gdf_7d_all)]:
    if len(df) > 0:
        within_bbox = (
            (df['longitude'] >= kal_bbox[0] - 0.1) &
            (df['longitude'] <= kal_bbox[2] + 0.1) &
            (df['latitude'] >= kal_bbox[1] - 0.1) &
            (df['latitude'] <= kal_bbox[3] + 0.1)
        ).all()
        assert_check(f"All {lbl} detections within Kalimantan bounding box", within_bbox)

# 4. Statistical & Monotonicity Checks
assert_check("24h Operational <= 24h Raw", len(gdf_24h) <= len(gdf_24h_all))
assert_check("72h Operational <= 72h Raw", len(gdf_72h) <= len(gdf_72h_all))
assert_check("7d Operational <= 7d Raw", len(gdf_7d) <= len(gdf_7d_all))

# Monotonicity check across widening windows (with small tolerance for boundary daily composites)
assert_check("Monotonicity: 24h Operational <= 72h Operational", len(gdf_24h) <= len(gdf_72h) + 1)
assert_check("Monotonicity: 72h Operational <= 7d Operational", len(gdf_72h) <= len(gdf_7d) + 1)

# Province total reconciliation
if len(gdf_7d) > 0:
    prov_sum = summary_province['7d'].sum()
    assert_check("Province 7d sum reconciles with operational 7d count", abs(prov_sum - len(gdf_7d)) <= 1)

print("-" * 70)
print(f"Validation Suite Summary: {val_passed} PASSED, {val_failed} FAILED")
if val_failed == 0:
    print("ALL VALIDATION CRITERIA SATISFIED.")
else:
    print("WARNING: Some validation checks failed. Inspect logs above.")
print("=" * 70)

## 14 — Methodological Limitations

The following scientific and operational constraints must be considered when interpreting outputs:

1. **Active-fire detection != Confirmed forest fire**: VIIRS detects mid-infrared thermal anomalies. These include biomass burning, agricultural clearance, industrial flares, and heat-reflective surfaces.
2. **Cloud and smoke obscuration**: Thick cloud cover or heavy smoke plumes can attenuate thermal radiation, leading to false negatives (missed detections).
3. **Multiplicity of detections**: A single large physical fire may trigger multiple contiguous 375 m pixel detections across consecutive overpasses and between SNPP and NOAA-20 platforms.
4. **Detection count != Burned area**: Active-fire counts represent instantaneous thermal detections, not cumulative area burned.
5. **Spatial resolution & centroid representation**: Detections are represented by 375 m pixel centroids rather than exact point GPS coordinates of combustion.
6. **Near Real-Time (NRT) status**: NRT products prioritize rapid latency over science-quality definitive calibration and may undergo subsequent reprocessing.
7. **Satellite observation overlap**: SNPP and NOAA-20 share similar orbital characteristics with staggered overpass times; detections from both cannot be summed as independent fire events.
8. **Administrative boundary precision**: geoBoundaries datasets provide general boundary references and do not constitute official Indonesian cadastral or legal demarcation.

## 15 — Analysis Metadata & Environment Log
Final run summary and software environment information.

In [ ]:
print("=" * 70)
print("ANALYSIS RUN METADATA")
print("=" * 70)
print(json.dumps(metadata, indent=2, default=str))

print("\n--- Software Environment ---")
print(f"Python      : {sys.version.split()[0]}")
print(f"Earth Engine: {ee.__version__}")
print(f"GeoPandas   : {gpd.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"Folium      : {folium.__version__}")

print("\n--- Exported Files Overview ---")
for base_dir in export_dirs:
    print(f"Directory: {base_dir}")
    if os.path.exists(base_dir):
        for root, dirs, files in os.walk(base_dir):
            lvl = root.replace(base_dir, '').count(os.sep)
            indent = ' ' * 4 * lvl
            sub = os.path.basename(root)
            if sub:
                print(f"{indent}* {sub}/")
            for f in files:
                f_p = os.path.join(root, f)
                print(f"{indent}   - {f} ({os.path.getsize(f_p):,} bytes)")
    print()

print("=" * 70)
print("KALIMANTAN FIRE SITUATION MONITOR: EXECUTION COMPLETE")
print("=" * 70)